# KINNEWS – GPU experiments (Colab / Kaggle)

Runs the neural experiments: BiLSTM (E7–E9) and Transformer fine-tuning (T1–T5).
**Runtime → Change runtime type → T4 GPU** before starting. Total time ≈ 45–60 min.

At the end, `results.zip` is downloaded — unzip it into the repo root locally so the
results sit next to the baseline results.

In [ ]:
REPO = "https://github.com/Samkwizera/NLP-application.git"   # <-- your repo URL
!git clone -q $REPO repo
%cd repo
!pip install -q -r requirements-train.txt
!python scripts/prepare_data.py
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime!")

## BiLSTM experiments (course: embeddings, RNN/LSTM, attention)
- **E7** random embeddings, max pooling
- **E8** Word2Vec-initialised embeddings, max pooling → does unsupervised pretraining on the same small corpus help?
- **E9** Word2Vec + additive attention pooling → does attention help on long articles?

In [ ]:
!python scripts/train_bilstm.py --run-name e7_bilstm_random_max --pooling max
!python scripts/train_bilstm.py --run-name e8_bilstm_w2v_max --pooling max --w2v
!python scripts/train_bilstm.py --run-name e9_bilstm_w2v_attention --pooling attention --w2v

## Transformer fine-tuning
Research question: does pretraining that *includes Kinyarwanda* beat general multilingual pretraining,
especially on the cross-source test set?
- **T1** `xlm-roberta-base` – general multilingual model (100 languages)
- **T2** `Davlan/afro-xlmr-base` – XLM-R further pretrained (MLM) on 17 African languages incl. Kinyarwanda
- **T3** `castorini/afriberta_base` – trained from scratch on 11 African languages incl. Kinyarwanda/Kirundi

In [ ]:
!python scripts/train_transformer.py --model FacebookAI/xlm-roberta-base --run-name t1_xlmr_base
!python scripts/train_transformer.py --model Davlan/afro-xlmr-base --run-name t2_afroxlmr_base
!python scripts/train_transformer.py --model castorini/afriberta_base --run-name t3_afriberta_base

**T4** – context length: the median article is ~300 words (more subword tokens), so 256 tokens truncates many.
Set `BEST` to the best of T1–T3 by **validation** macro-F1 (never pick by test).

In [ ]:
import json, glob
for f in sorted(glob.glob("results/t*.json")):
    d = json.load(open(f)); print(f"{d['name']:25s} val mF1 {d['val']['macro_f1']:.4f}  test mF1 {d['test']['macro_f1']:.4f}")

In [ ]:
BEST = "Davlan/afro-xlmr-base"   # <-- edit after looking at val scores above
!python scripts/train_transformer.py --model $BEST --run-name t4_best_maxlen512 --max-length 512 --batch-size 8 --save-model

**T5** - same as T4 but with balanced class weights in the loss, to see if it helps the rare topics like it did for TF-IDF (E5).

In [ ]:
!python scripts/train_transformer.py --model $BEST --run-name t5_best_maxlen512_weighted --max-length 512 --batch-size 8 --class-weights --save-model

## Download results

In [ ]:
!zip -qr results.zip results
from google.colab import files; files.download("results.zip")

## Publish the final model to the Hugging Face Hub (used by the web app)
Create a **write** token at https://huggingface.co/settings/tokens first.

In [ ]:
from huggingface_hub import login; login()
from transformers import AutoModelForSequenceClassification, AutoTokenizer
path = "models/t4_best_maxlen512/final"
HUB_ID = "Samkwizera/kinnews-topic-classifier"   # <-- your HF username
AutoModelForSequenceClassification.from_pretrained(path).push_to_hub(HUB_ID)
AutoTokenizer.from_pretrained(path).push_to_hub(HUB_ID)